In [1]:
#steg 1: importera numpy och ladda data

import numpy as np
#print("Numpy fungerar!")

#ladda data

data = np.load("credit_score_fairness_data.npy")
print(data)

[[0. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 ...
 [1. 1. 1.]
 [0. 1. 1.]
 [0. 1. 1.]]


In [2]:
#Steg 2: dela upp data i tre olika arrayer

protected_attribute = data[:, 0].astype(int)
true_credit_worthiness = data[:, 1].astype(int)
algorithm_prediction = data[:, 2].astype(int)

print(protected_attribute)
print(true_credit_worthiness)
print(algorithm_prediction)


[0 1 1 ... 1 0 0]
[1 1 1 ... 1 1 1]
[1 1 1 ... 1 1 1]


In [6]:
#steg 3:Calculate equal opportunity and equalized error rates by calculating the elements of a confusion matrix. Does the algorithm predict credit scores fairly?
#För att beräkna equal opportunity och equalized error rates behöver vi först skapa en confusion matrix för varje grupp (de som tillhör den skyddade gruppen och de som inte gör det).

#"bra credit score" = 1, "dålig credit score" = 0
#TP=true positive, FP=false positive, TN=true negative, FN=false negative

def confusion_matrix(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    return TP, FP, TN, FN

#om division med noll sker, returnera 0 istället för att ge ett fel
def safe_division(n, d):
    return n / d if d != 0 else np.nan

#beräkna rates för grupperna
def calculate_rates(protected_attribute, true_credit_worthiness, algorithm_prediction):
    #grupp 1: skyddad grupp (protected_attribute == 1)
    protected_group = protected_attribute == 1      #skyddad grupp (grupp 1)
    TP_protected, FP_protected, TN_protected, FN_protected = confusion_matrix(true_credit_worthiness[protected_group], algorithm_prediction[protected_group])
    
    #grupp 2: icke-skyddad grupp (protected_attribute == 0)
    non_protected_group = protected_attribute == 0      #oskyddad grupp, (grupp 2)
    TP_non_protected, FP_non_protected, TN_non_protected, FN_non_protected = confusion_matrix(true_credit_worthiness[non_protected_group], algorithm_prediction[non_protected_group])
    
    #beräkna equal opportunity (TPR) och equalized error rates (FPR) för båda grupperna
    #beräkna för grupp 1 (skyddad grupp)
    TPR_protected = safe_division(TP_protected, TP_protected + FN_protected)
    FPR_protected = safe_division(FP_protected, FP_protected + TN_protected)
    FNR_protected = safe_division(FN_protected, TP_protected + FN_protected)
    
    #beräkna för grupp 2 (icke-skyddad grupp)
    TPR_non_protected = safe_division(TP_non_protected, TP_non_protected + FN_non_protected)
    FPR_non_protected = safe_division(FP_non_protected, FP_non_protected + TN_non_protected)
    FNR_non_protected = safe_division(FN_non_protected, TP_non_protected + FN_non_protected)    
    
    return TPR_protected, FPR_protected, FNR_protected, TPR_non_protected, FPR_non_protected, FNR_non_protected

TPR_protected, FPR_protected, FNR_protected, TPR_non_protected, FPR_non_protected, FNR_non_protected = calculate_rates(protected_attribute, true_credit_worthiness, algorithm_prediction)




In [ ]:
print(f"Skyddad grupp - TPR: {TPR_protected:.2f}, FPR: {FPR_protected:.2f}, FNR: {FNR_protected:.2f}")
print(f"Icke-skyddad grupp - TPR: {TPR_non_protected:.2f}, FPR: {FPR_non_protected:.2f}, FNR: {FNR_non_protected:.2f}")

# Enkel utvärdering
threshold=0.05
diff_tpr = abs(TPR_protected - TPR_non_protected)
diff_fpr = abs(FPR_protected - FPR_non_protected)
diff_fnr = abs(FNR_protected - FNR_non_protected)

print(f'diff_tpr: {diff_tpr:.2f}')
print(f'diff_fpr: {diff_fpr:.2f}')
print(f'diff_fnr: {diff_fnr:.2f}')

if diff_tpr < threshold and diff_fpr < threshold and diff_fnr < threshold:
    print("Algoritmen verkar uppfylla Equal Opportunity.")
else:
    print("Algoritmen verkar inte uppfylla Equal Opportunity.")
    #vilken grupp är gynnad?
    # if TPR_protected > TPR_non_protected:
    #     print("Den skyddade gruppen är gynnad när det gäller TPR.")
    # else:
    #     print("Den icke-skyddade gruppen är gynnad när det gäller TPR.")

    # if FPR_protected < FPR_non_protected:
    #     print("Den skyddade gruppen är gynnad när det gäller FPR.")
    # else:
    #     print("Den icke-skyddade gruppen är gynnad när det gäller FPR.")

    # if FNR_protected < FNR_non_protected:
    #     print("Den skyddade gruppen är gynnad när det gäller FNR.")
    # else:
    #     print("Den icke-skyddade gruppen är gynnad när det gäller FNR.")    


    

Skyddad grupp - TPR: 0.79, FPR: 0.29, FNR: 0.21
Icke-skyddad grupp - TPR: 0.81, FPR: 0.09, FNR: 0.19
diff_tpr: 0.02
diff_fpr: 0.21
diff_fnr: 0.02
Algoritmen verkar inte uppfylla Equal Opportunity.


In [5]:
# Does the algorithm predict credit scores fairly?
    # Om TPR (True Positive Rate) är lika eller mycket nära för både den skyddade och icke-skyddade gruppen, kan vi säga att algoritmen uppfyller Equal Opportunity. I det här fallet, om skillnaden i TPR är mindre än 0.05, indikerar det att algoritmen inte diskriminerar mellan grupperna när det gäller att korrekt identifiera de som är kreditvärdiga.
    #I equalize error rates, om FPR (False Positive Rate) är lika eller mycket nära för både grupperna, kan vi säga att algoritmen uppfyller Equalized Error Rates. I det här fallet, om skillnaden i FPR är mindre än 0.05, indikerar det att algoritmen inte diskriminerar mellan grupperna när det gäller att felaktigt identifiera de som inte är kreditvärdiga som kreditvärdiga.
    #FPR för den skyddade grupen är 0.29 och för den icke-skyddade gruppen är den 0.09. Detta innebär att icke-kreditvärdiga personer i den skyddade gruppen felaktigt kallifieras.

# Which fairness metric do you find most suitable to consider for this specific case? 
    #equal opportunity (TPR) är mer lämplig i det här fallet eftersom det fokuserar på att säkerställa att de som är kreditvärdiga (true positives) behandlas lika oavsett grupp. Det är viktigt att algoritmen inte diskriminerar mot den skyddade gruppen när det gäller att korrekt identifiera de som är kreditvärdiga, eftersom det kan ha en direkt påverkan på människors liv och ekonomiska möjligheter.
